<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_08_model_tuning/stage_08_02_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_08_02 - TUNING - XGBOOST**

# **SETUP COMÚN DEL PIPELINE DE ENTRENAMIENTO**

## **1. Imports**

In [ ]:
# Permite anotaciones modernas en Python < 3.11
from __future__ import annotations

# ================================
# Standard library
# ================================
import os
import sys
import json
import time
import random
import importlib
import warnings
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Any, Dict, List, Iterable, Tuple

# ================================
# Third-party
# ================================
import numpy as np
import joblib

# ================================
# Configuración global
# ================================

# ---- Warnings (controlado, no agresivo)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

# ---- Logging limpio para notebooks
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)
logger.propagate = False  # evita duplicación con el root logger

# limpiar handlers si se re-ejecuta la celda
if logger.handlers:
    logger.handlers.clear()

handler = logging.StreamHandler()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
handler.setFormatter(formatter)
logger.addHandler(handler)

logger.info("Environment initialized")

2026-04-08 18:18:44,990 | INFO | Environment initialized


## **2. Acceso a drive**

In [ ]:
# ================================
# Entorno (Google Drive / local)
# ================================

from pathlib import Path
import os

# Detectar si estamos en Colab
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    DEFAULT_DRIVE_DIR = "/content/drive/MyDrive/neural_profit/"
else:
    # fallback local (puedes ajustarlo si quieres)
    DEFAULT_DRIVE_DIR = "./neural_profit/"

# Ruta base del proyecto
DRIVE_DIR = Path(os.environ.get("DRIVE_DIR", DEFAULT_DRIVE_DIR))

# Validación básica
if not DRIVE_DIR.exists():
    logger.warning(f"DRIVE_DIR no existe: {DRIVE_DIR}")
else:
    logger.info(f"DRIVE_DIR: {DRIVE_DIR}")

2026-04-08 18:18:47,411 | INFO | DRIVE_DIR: /content/drive/MyDrive/neural_profit


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **3. Rutas de ventanas `seq2one` y `scaler`**

In [ ]:
# ================================
# Configuración de paths
# ================================

WINDOWS_SEQ2ONE_DIR = DRIVE_DIR / os.environ.get(
    "WINDOWS_SEQ2ONE_DIR",
    "data/07_windows/seq2one/"
)

SCALERS_DIR = DRIVE_DIR / os.environ.get(
    "SCALERS_DIR",
    "data/06_scaled/"
)

# Validación básica
for p in [WINDOWS_SEQ2ONE_DIR, SCALERS_DIR]:
    if not p.exists():
        logger.warning(f"Path no existe: {p}")
    else:
        logger.info(f"Path OK: {p}")

# ================================
# Configuración del experimento
# ================================

# Targets T2 (clasificación)
TARGETS = [
    "t2_dir_thr_90",
    "t2_dir_thr_120",
]

# Tamaños de ventana
WINDOW_SIZES = [30, 60, 90, 120, 180]

# Splits
SPLITS = ["train", "valid", "test"]

# Features finales (consistencia global)
FEATURES_T2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]

logger.info("Configuración de experimento cargada")
logger.info(f"Targets: {TARGETS}")
logger.info(f"Window sizes: {WINDOW_SIZES}")

2026-04-08 18:18:47,424 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/07_windows/seq2one
2026-04-08 18:18:47,425 | INFO | Path OK: /content/drive/MyDrive/neural_profit/data/06_scaled
2026-04-08 18:18:47,427 | INFO | Configuración de experimento cargada
2026-04-08 18:18:47,427 | INFO | Targets: ['t2_dir_thr_90', 't2_dir_thr_120']
2026-04-08 18:18:47,428 | INFO | Window sizes: [30, 60, 90, 120, 180]


In [ ]:
# ================================
# Construcción y validación de paths (windows + scaler compartido)
# ================================

def build_windows_paths(window_sizes, targets, splits, base_dir):
    """
    windows_paths[w][t][s] -> Path
    """
    windows_paths = {}

    for w in window_sizes:
        windows_paths[w] = {}
        for t in targets:
            windows_paths[w][t] = {}
            for s in splits:
                windows_paths[w][t][s] = (
                    base_dir
                    / f"L{w}"
                    / f"windows_{t}_{s}.npz"
                )

    return windows_paths


def build_shared_scaler_path(base_dir, scaler_name="scaler_t2.pkl"):
    """
    Retorna el path del scaler compartido para T2.
    """
    return base_dir / scaler_name


# ================================
# Construcción
# ================================

WINDOWS_PATHS = build_windows_paths(
    window_sizes=WINDOW_SIZES,
    targets=TARGETS,
    splits=SPLITS,
    base_dir=WINDOWS_SEQ2ONE_DIR,
)

SCALER_T2_PATH = build_shared_scaler_path(
    base_dir=SCALERS_DIR,
    scaler_name="scaler_t2.pkl",
)

logger.info("Paths construidos (windows + scaler compartido T2)")

# ================================
# Validación
# ================================

missing_windows = []
existing_windows = []

for w in WINDOW_SIZES:
    for t in TARGETS:
        for s in SPLITS:
            p = WINDOWS_PATHS[w][t][s]
            if p.exists():
                existing_windows.append(p)
            else:
                missing_windows.append(p)

scaler_exists = SCALER_T2_PATH.exists()

# -------------------------------
# Logs
# -------------------------------
logger.info(f"Windows OK      : {len(existing_windows)}")
logger.info(f"Windows missing : {len(missing_windows)}")

if scaler_exists:
    logger.info(f"Scaler T2 OK    : {SCALER_T2_PATH}")
else:
    logger.warning(f"Scaler T2 missing: {SCALER_T2_PATH}")

if missing_windows:
    logger.warning("Archivos de ventanas faltantes:")
    for p in missing_windows[:5]:
        logger.warning(f"Missing window: {p}")

if not missing_windows and scaler_exists:
    logger.info("Todos los archivos existen correctamente")

# ================================
# Ejemplo de uso
# ================================

# window
# WINDOWS_PATHS[60]["t2_dir_thr_90"]["train"]

# scaler compartido
# SCALER_T2_PATH

2026-04-08 18:18:47,442 | INFO | Paths construidos (windows + scaler compartido T2)
2026-04-08 18:18:47,454 | INFO | Windows OK      : 30
2026-04-08 18:18:47,455 | INFO | Windows missing : 0
2026-04-08 18:18:47,457 | INFO | Scaler T2 OK    : /content/drive/MyDrive/neural_profit/data/06_scaled/scaler_t2.pkl
2026-04-08 18:18:47,458 | INFO | Todos los archivos existen correctamente


## **4. Carga de ventanas X/y**

### **4.1. Cargar ventanas (*.npz)**

In [ ]:
from pathlib import Path
from typing import Tuple
import numpy as np


def load_npz_windows(path: Path) -> Tuple[np.ndarray, np.ndarray]:
    """
    Carga ventanas X e y desde un archivo .npz estándar (T2 seq2one).

    Espera:
    - 'X': (n_samples, seq_len, n_features)
    - 'y' o 'Y': (n_samples,) o (n_samples, 1) o (n_samples, seq_len, 1)

    Retorna
    -------
    X : np.ndarray  -> (n_samples, seq_len, n_features)
    y : np.ndarray  -> (n_samples,)
    """

    # --------------------------------------------------
    # 1. Validación
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    with np.load(path) as data:

        if "X" not in data:
            raise KeyError(f"NPZ inválido (falta clave 'X'): {path}")

        X = data["X"]

        if "y" in data:
            y = data["y"]
        elif "Y" in data:
            y = data["Y"]
        else:
            raise KeyError(f"NPZ inválido (falta clave 'y'/'Y'): {path}")

        # copiar a memoria
        X = X.copy()
        y = y.copy()

    # --------------------------------------------------
    # 3. Ajuste de shape de y (CRÍTICO)
    # --------------------------------------------------
    # Caso (n_samples, 1)
    if y.ndim == 2 and y.shape[1] == 1:
        y = y.reshape(-1)

    # Caso (n_samples, seq_len, 1) -> tomar último valor
    elif y.ndim == 3:
        y = y[:, -1, 0]

    # Caso (n_samples, seq_len) -> tomar último valor
    elif y.ndim == 2 and y.shape[1] > 1:
        y = y[:, -1]

    # Validación final
    if y.ndim != 1:
        raise ValueError(f"y no es 1D después de procesamiento: shape={y.shape}")

    # --------------------------------------------------
    # 4. Logs útiles
    # --------------------------------------------------
    logger.info(f"Loaded: {path.name}")
    logger.info(f"X shape: {X.shape} | y shape: {y.shape}")

    return X, y

### **4.2. Cargar escalador (*.pkl)**

In [ ]:
# --------------------------------------------------
# Función común: carga scaler compartido (T2)
# --------------------------------------------------
def load_scaler(path: Path) -> Any:
    """
    Carga un scaler serializado con joblib.

    Parámetros
    ----------
    path : Path
        Ruta al archivo scaler (.pkl)

    Retorna
    -------
    scaler : Any
        Objeto scaler (ej. StandardScaler, MinMaxScaler)
    """

    # --------------------------------------------------
    # 1. Validación de existencia
    # --------------------------------------------------
    if not path.exists():
        raise FileNotFoundError(f"No se encontró el scaler: {path}")

    # --------------------------------------------------
    # 2. Carga
    # --------------------------------------------------
    scaler = joblib.load(path)

    # --------------------------------------------------
    # 3. Validación básica (opcional pero útil)
    # --------------------------------------------------
    if not hasattr(scaler, "transform"):
        raise TypeError(f"El objeto cargado no es un scaler válido: {type(scaler)}")

    # --------------------------------------------------
    # 4. Logging
    # --------------------------------------------------
    logger.info(f"Scaler cargado: {path.name}")

    return scaler

### **4.3. Función de carga**

In [ ]:
from typing import Any, Dict, Mapping
from pathlib import Path


# --------------------------------------------------
# Carga completa: ventanas + scaler compartido T2
# --------------------------------------------------
def load_windows_and_scaler(
    *,
    window_size: int,
    target: str,
    windows_paths: Mapping[int, Mapping[str, Mapping[str, Path]]],
    scaler_path: Path,
) -> Dict[str, Any]:
    """
    Carga X/y (train/valid/test) y el scaler compartido T2
    para un par (window_size, target).

    Parámetros
    ----------
    window_size : int
        Tamaño de ventana.
    target : str
        Target T2, por ejemplo:
        - 't2_dir_thr_90'
        - 't2_dir_thr_120'
    windows_paths : mapping
        Estructura:
        windows_paths[window_size][target][split] -> Path
    scaler_path : Path
        Ruta al scaler compartido T2.

    Retorna
    -------
    bundle : dict
        Diccionario con:
        - metadata
        - paths
        - scaler
        - train/valid/test con X e y
    """

    # --------------------------
    # 1) Validaciones
    # --------------------------
    if window_size not in windows_paths:
        raise KeyError(f"window_size={window_size} no existe en windows_paths")

    if target not in windows_paths[window_size]:
        raise KeyError(f"target='{target}' no existe en windows_paths[{window_size}]")

    for split in ["train", "valid", "test"]:
        if split not in windows_paths[window_size][target]:
            raise KeyError(
                f"split='{split}' no existe en windows_paths[{window_size}]['{target}']"
            )

    if not scaler_path.exists():
        raise FileNotFoundError(f"No se encontró el scaler compartido: {scaler_path}")

    # --------------------------
    # 2) Paths
    # --------------------------
    train_path = windows_paths[window_size][target]["train"]
    valid_path = windows_paths[window_size][target]["valid"]
    test_path  = windows_paths[window_size][target]["test"]

    # --------------------------
    # 3) Carga de ventanas
    # --------------------------
    X_train, y_train = load_npz_windows(train_path)
    X_valid, y_valid = load_npz_windows(valid_path)
    X_test, y_test   = load_npz_windows(test_path)

    # --------------------------
    # 4) Carga de scaler
    # --------------------------
    scaler = load_scaler(scaler_path)

    # --------------------------
    # 5) Inferir horizonte
    # --------------------------
    try:
        horizon = int(target.split("_")[-1])
    except Exception:
        horizon = None

    # --------------------------
    # 6) Logging
    # --------------------------
    logger.info(
        f"Bundle cargado | target={target} | window_size={window_size} | "
        f"train={X_train.shape} | valid={X_valid.shape} | test={X_test.shape}"
    )

    # --------------------------
    # 7) Retorno
    # --------------------------
    return {
        "window_size": window_size,
        "target": target,
        "horizon": horizon,
        "paths": {
            "train": str(train_path),
            "valid": str(valid_path),
            "test": str(test_path),
            "scaler": str(scaler_path),
        },
        "scaler": scaler,
        "train": {
            "X": X_train,
            "y": y_train,
        },
        "valid": {
            "X": X_valid,
            "y": y_valid,
        },
        "test": {
            "X": X_test,
            "y": y_test,
        },
    }


### **4.4. Creación de bundles T2**

In [ ]:
# --------------------------------------------------
# Crea bundles T2 para un window_size dado
# --------------------------------------------------
def create_bundles(
    window_size: int,
    targets: list[str] = TARGETS,
    windows_paths=WINDOWS_PATHS,
    scaler_path=SCALER_T2_PATH,
):
    """
    Crea un bundle por target para un window_size dado.

    Retorna
    -------
    bundle_t2_90  : dict
    bundle_t2_120 : dict
    """

    if len(targets) != 2:
        raise ValueError(
            f"Se esperaban exactamente 2 targets T2. Recibido: {targets}"
        )

    bundle_t2_90 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[0],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    bundle_t2_120 = load_windows_and_scaler(
        window_size=window_size,
        target=targets[1],
        windows_paths=windows_paths,
        scaler_path=scaler_path,
    )

    # --------------------------------------------------
    # Verificación rápida
    # --------------------------------------------------
    print(f"\n{'=' * 70}")
    print(f"WINDOW_SIZE: {window_size}")
    print(f"{'=' * 70}")

    print(f"\nTARGET: {targets[0]}")
    print("Train :", bundle_t2_90["train"]["X"].shape, bundle_t2_90["train"]["y"].shape)
    print("Valid :", bundle_t2_90["valid"]["X"].shape, bundle_t2_90["valid"]["y"].shape)
    print("Test  :", bundle_t2_90["test"]["X"].shape,  bundle_t2_90["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_90["scaler"]).__name__)

    print(f"\nTARGET: {targets[1]}")
    print("Train :", bundle_t2_120["train"]["X"].shape, bundle_t2_120["train"]["y"].shape)
    print("Valid :", bundle_t2_120["valid"]["X"].shape, bundle_t2_120["valid"]["y"].shape)
    print("Test  :", bundle_t2_120["test"]["X"].shape,  bundle_t2_120["test"]["y"].shape)
    print("Scaler:", type(bundle_t2_120["scaler"]).__name__)

    return bundle_t2_90, bundle_t2_120

In [ ]:
#bundle_t2_90, bundle_t2_120 = create_bundles(window_size=60)

Como acceder a las ventanas X e y:

```python
X_train_90 = bundle_t2_90["train"]["X"]
y_train_90 = bundle_t2_90["train"]["y"]

X_valid_120 = bundle_t2_120["valid"]["X"]
y_valid_120 = bundle_t2_120["valid"]["y"]

scaler = bundle_t2_90["scaler"]
```




### **4.5. Preparación de inputs según el tipo de modelo**

In [ ]:
# ============================================================
# Preparación de inputs según el tipo de modelo
# ============================================================

def flatten_seq2one_X(X: np.ndarray) -> np.ndarray:
    """
    Aplana ventanas seq2one para modelos tabulares.

    Convierte:
        (n_samples, seq_len, n_features)
    a:
        (n_samples, seq_len * n_features)
    """
    X = np.asarray(X)

    if X.ndim != 3:
        raise ValueError(
            f"Se esperaba X 3D con shape (n, seq_len, n_features). "
            f"Recibido X.shape={X.shape}"
        )

    n_samples = X.shape[0]
    return X.reshape(n_samples, -1)


def prepare_X_for_model(
    X: np.ndarray,
    *,
    input_mode: str,
) -> np.ndarray:
    """
    Prepara X según el tipo de modelo.

    Parámetros
    ----------
    X : np.ndarray
        Array de entrada.
    input_mode : str
        - '2d_flat' : aplana ventanas 3D a 2D
        - '3d'      : deja X como está

    Retorna
    -------
    np.ndarray
        X transformado para el modelo correspondiente.
    """
    X = np.asarray(X)

    if input_mode == "3d":
        if X.ndim != 3:
            raise ValueError(
                f"Se esperaba X 3D para input_mode='3d'. "
                f"Recibido X.shape={X.shape}"
            )
        return X

    if input_mode == "2d_flat":
        return flatten_seq2one_X(X)

    raise ValueError(
        f"input_mode no soportado: {input_mode}. "
        f"Use '2d_flat' o '3d'."
    )

**Cómo se usa después**

Para Logistic Regression:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="2d_flat")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="2d_flat")
```

Para LSTM / GRU / Transformer:

```python
X_train_model = prepare_X_for_model(bundle["train"]["X"], input_mode="3d")
X_valid_model = prepare_X_for_model(bundle["valid"]["X"], input_mode="3d")
```


## **5. Sanity Check**

In [ ]:
from __future__ import annotations

from typing import Any, Optional, Tuple
import numpy as np


# ============================================================
# 1) Utilidades internas (inferencias y normalización)
# ============================================================

def _as_numpy_array(a: Any, *, name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray y valida que no esté vacía.
    """
    arr = np.asarray(a)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío. shape={arr.shape}")

    if not np.isfinite(arr).all():
        raise ValueError(
            f"{name} contiene NaN/inf. Limpie antes de continuar. shape={arr.shape}"
        )

    return arr


def _normalize_y_seq2one(y: np.ndarray, *, name: str = "y") -> np.ndarray:
    """
    Normaliza y para seq2one a shape (n_samples,).

    Acepta:
    - (n,)
    - (n, 1)

    Rechaza shapes incompatibles.
    """
    if y.ndim == 1:
        return y

    if y.ndim == 2 and y.shape[1] == 1:
        return y.squeeze(1)

    raise ValueError(
        f"{name} shape inválido para seq2one. "
        f"Se esperaba (n,) o (n,1). Recibido {y.shape}"
    )


def _infer_seq_len_and_n_features(X: np.ndarray) -> Tuple[int, Optional[int], str]:
    """
    Infiere (seq_len, n_features, mode) desde X.

    mode:
    - '3d': X = (n, seq_len, n_features)
    - '2d': X = (n, d_flat)  -> n_features=None
    """
    if X.ndim == 3:
        return int(X.shape[1]), int(X.shape[2]), "3d"

    if X.ndim == 2:
        return int(X.shape[1]), None, "2d"

    raise ValueError(
        f"X.ndim inválido. Se esperaba 2D o 3D. Recibido X.shape={X.shape}"
    )

In [ ]:
# ============================================================
# 2) Sanity check principal (seq2one clasificación T2)
# ============================================================

def sanity_check_seq2one(
    X: Any,
    y: Any,
    split_name: str,
    *,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    allow_seq_inputs_take_last: bool = False,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Sanity check para datasets seq2one de clasificación.

    Soporta:
      - X 3D: (n, seq_len, n_features)
      - X 2D: (n, d_flat)  [ventana aplanada]

    y soporta:
      - y 1D: (n,)
      - y 2D: (n,1)

    Parámetros
    ----------
    expected_seq_len:
        Para X 3D: valida seq_len.
        Para X 2D: si expected_flat_dim no está, puede usarse como d_flat esperado.
    expected_n_features:
        Solo aplica a X 3D.
    expected_flat_dim:
        Solo aplica a X 2D (d_flat esperado).
    allow_seq_inputs_take_last:
        Si y viniera como (n, seq_len), permite tomar y[:, -1].
        Por defecto False.
    """

    X = _as_numpy_array(X, name=f"X[{split_name}]")
    y = _as_numpy_array(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Normalización de y
    # --------------------------------------------------
    if y.ndim == 2 and y.shape[1] != 1 and allow_seq_inputs_take_last:
        y = y[:, -1]

    y = _normalize_y_seq2one(y, name=f"y[{split_name}]")

    # --------------------------------------------------
    # Inferir modo y dimensiones de X
    # --------------------------------------------------
    seq_len, n_features, mode = _infer_seq_len_and_n_features(X)
    n_samples = int(X.shape[0])

    # --------------------------------------------------
    # Validación básica de n_samples
    # --------------------------------------------------
    if y.shape[0] != n_samples:
        raise ValueError(
            f"Mismatch n_samples en {split_name}: "
            f"X tiene n={n_samples}, y tiene n={y.shape[0]}"
        )

    # --------------------------------------------------
    # Validación de shapes según modo
    # --------------------------------------------------
    if mode == "3d":
        if expected_seq_len is not None and seq_len != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: seq_len esperado={expected_seq_len}, "
                f"recibido={seq_len}. X.shape={X.shape}"
            )

        if expected_n_features is not None and n_features != int(expected_n_features):
            raise ValueError(
                f"{split_name}: n_features esperado={expected_n_features}, "
                f"recibido={n_features}. X.shape={X.shape}"
            )

    elif mode == "2d":
        d_flat = int(X.shape[1])

        if expected_flat_dim is not None and d_flat != int(expected_flat_dim):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_flat_dim}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_flat_dim is None and expected_seq_len is not None and d_flat != int(expected_seq_len):
            raise ValueError(
                f"{split_name}: d_flat esperado={expected_seq_len}, "
                f"recibido={d_flat}. X.shape={X.shape}"
            )

        if expected_n_features is not None:
            raise ValueError(
                f"{split_name}: expected_n_features no aplica para X 2D. "
                f"Use expected_flat_dim o pase X en 3D."
            )

    # --------------------------------------------------
    # Diagnóstico de clases
    # --------------------------------------------------
    classes, counts = np.unique(y, return_counts=True)
    class_distribution = {
        str(cls): int(cnt) for cls, cnt in zip(classes, counts)
    }

    if len(classes) < 2:
        raise ValueError(
            f"{split_name}: y contiene menos de 2 clases únicas. "
            f"classes={classes.tolist()}"
        )

    # --------------------------------------------------
    # Salida informativa
    # --------------------------------------------------
    info = {
        "split": split_name,
        "X_shape": tuple(X.shape),
        "y_shape": tuple(y.shape),
        "mode": mode,
        "seq_len": seq_len if mode == "3d" else None,
        "n_features": n_features if mode == "3d" else None,
        "flat_dim": int(X.shape[1]) if mode == "2d" else None,
        "n_classes": int(len(classes)),
        "classes": classes.tolist(),
        "class_distribution": class_distribution,
    }

    if verbose:
        print(
            f"[sanity_check_seq2one] {split_name} | "
            f"X={info['X_shape']} | y={info['y_shape']} | "
            f"mode={mode} | n_classes={info['n_classes']} | "
            f"classes={info['classes']}"
        )

    return info

In [ ]:
# ============================================================
# 3) Sanity checks por bundle (train/valid/test)
# ============================================================

def run_sanity_checks_for_bundle_seq2one(
    bundle: Dict[str, Any],
    *,
    tag: str,
    expected_seq_len: Optional[int] = None,
    expected_n_features: Optional[int] = None,
    expected_flat_dim: Optional[int] = None,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Ejecuta sanity checks para un bundle con estructura:

    {
      "target": "t2_dir_thr_90",
      "horizon": 90,
      "train": {"X":..., "y":...},
      "valid": {"X":..., "y":...},
      "test":  {"X":..., "y":...},
    }

    Detecta automáticamente si X es 2D o 3D, y usa TRAIN como referencia
    si no se pasan expected_*.
    """
    X_tr = bundle["train"]["X"]
    y_tr = bundle["train"]["y"]
    X_va = bundle["valid"]["X"]
    y_va = bundle["valid"]["y"]
    X_te = bundle["test"]["X"]
    y_te = bundle["test"]["y"]

    X_tr_arr = np.asarray(X_tr)
    seq_len_tr, n_feat_tr, mode_tr = _infer_seq_len_and_n_features(X_tr_arr)

    # --------------------------------------------------
    # Setear esperados desde TRAIN si no se dieron
    # --------------------------------------------------
    if mode_tr == "3d":
        if expected_seq_len is None:
            expected_seq_len = seq_len_tr
        if expected_n_features is None:
            expected_n_features = n_feat_tr
        expected_flat_dim = None
    else:
        d_flat_tr = int(X_tr_arr.shape[1])
        if expected_flat_dim is None:
            expected_flat_dim = d_flat_tr
        expected_n_features = None

    # --------------------------------------------------
    # Ejecutar checks
    # --------------------------------------------------
    out_tr = sanity_check_seq2one(
        X_tr,
        y_tr,
        f"train_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_va = sanity_check_seq2one(
        X_va,
        y_va,
        f"valid_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    out_te = sanity_check_seq2one(
        X_te,
        y_te,
        f"test_{tag}",
        expected_seq_len=expected_seq_len,
        expected_n_features=expected_n_features,
        expected_flat_dim=expected_flat_dim,
        verbose=verbose,
    )

    target = bundle.get("target", "NA")
    horizon = bundle.get("horizon", "NA")

    if verbose:
        print(f"OK {tag} | target={target} | horizon={horizon}")

    return {
        "target": target,
        "horizon": horizon,
        "train": out_tr,
        "valid": out_va,
        "test": out_te,
    }


def run_sanity_checks_all_targets_seq2one(
    bundle_t2_90: Dict[str, Any],
    bundle_t2_120: Dict[str, Any],
    *,
    verbose: bool = True,
) -> Dict[str, Any]:
    """
    Corre sanity checks para ambos targets T2.
    """
    out_t2_90 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_90,
        tag="t2_90",
        verbose=verbose,
    )

    out_t2_120 = run_sanity_checks_for_bundle_seq2one(
        bundle_t2_120,
        tag="t2_120",
        verbose=verbose,
    )

    return {
        "t2_dir_thr_90": out_t2_90,
        "t2_dir_thr_120": out_t2_120,
    }

In [ ]:
#sanity_outputs = run_sanity_checks_all_targets_seq2one(
#    bundle_t2_90,
#    bundle_t2_120,
#    verbose=True,
#)

In [ ]:
#summary_delta = run_sanity_checks_all_horizons_seq2one(bundle_delta_60, bundle_delta_90)
#summary_ret = run_sanity_checks_all_horizons_seq2one(bundle_ret_60, bundle_ret_90)

## **6. Métricas de clasificación T2**

In [ ]:
# ================================
# Setup para importar módulos del proyecto
# ================================

if str(DRIVE_DIR) not in sys.path:
    sys.path.insert(0, str(DRIVE_DIR))

(DRIVE_DIR / "metrics" / "__init__.py").touch(exist_ok=True)

importlib.invalidate_caches()

# ================================
# Imports de métricas (T2 clasificación)
# ================================

from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

# ================================
# Utilidad: métricas → DataFrame
# ================================

import pandas as pd

def metrics_to_df(
    metrics: dict,
    *,
    model: str,
    split: str,
    window_size: int,
    target: str,
) -> pd.DataFrame:
    """
    Convierte métricas de clasificación T2 en una fila de DataFrame.
    """

    m = metrics["metrics"]

    return pd.DataFrame([{
        "model": model,
        "split": split,
        "window_size": window_size,
        "target": target,
        "n_samples": m["n_samples"],

        # principales
        "balanced_accuracy": m["balanced_accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],

        # complementarias
        "accuracy": m["accuracy"],
        "precision_macro": m["precision_macro"],
        "recall_macro": m["recall_macro"],

        # baseline
        "balanced_accuracy_naive": m.get("balanced_accuracy_naive"),
        "bal_acc_gain_vs_naive": m.get("balanced_accuracy_gain_vs_naive"),
    }])

logger.info("Métricas T2 + utilidades cargadas")

2026-04-08 18:18:47,609 | INFO | Métricas T2 + utilidades cargadas


## **7. Persistencia de métricas (tracking de experimentos)**

In [ ]:
# ================================
# Carga de métricas (si existen)
# ================================

def load_classification_metrics_if_exists(
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> pd.DataFrame:
    """
    Carga métricas de clasificación si el archivo existe.
    """

    path = base_dir / f"classification_{name}_metrics.parquet"

    if path.exists():
        logger.info(f"Cargando métricas desde: {path}")
        return pd.read_parquet(path)

    logger.info(f"No existen métricas previas para: {name}")
    return pd.DataFrame()

In [ ]:
# ================================
# Guardado de métricas
# ================================

def save_classification_metrics(
    df_metrics: pd.DataFrame,
    *,
    name: str,
    base_dir: Path = DRIVE_DIR / "metrics/classification_metrics",
) -> Path:
    """
    Guarda un DataFrame de métricas de clasificación en formato Parquet.
    """

    base_dir.mkdir(parents=True, exist_ok=True)

    out_path = base_dir / f"classification_{name}_metrics.parquet"

    df_metrics.to_parquet(out_path, index=False)

    logger.info(f"Métricas guardadas en: {out_path}")

    return out_path

Ejemplo de uso:

```python
df_metrics = load_classification_metrics_if_exists(name="lstm_valid")

df_metrics = pd.concat([df_metrics, new_row_df], ignore_index=True)

save_classification_metrics(df_metrics, name="lstm_valid")
```



## **8. Gestión de dispositivo y memoria**

In [ ]:
# ================================
# Device y limpieza de memoria
# ================================

def get_torch_device():
    """
    Retorna el device para modelos PyTorch.
    """
    import torch

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"Using device: {device}")
    return device


def clear_torch_memory() -> None:
    """
    Limpia memoria Python y, si existe CUDA, libera caché GPU.
    Útil entre entrenamientos de modelos PyTorch.
    """
    import gc

    gc.collect()

    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass

## **9. Reproducibilidad**

In [ ]:
# ================================
# Reproducibilidad
# ================================

def set_seeds(seed: int = 42) -> None:
    """
    Fija semillas para reproducibilidad en:
    - Python
    - NumPy
    - PyTorch (si está disponible)
    """

    import random
    import numpy as np
    import os

    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # ---- PyTorch (si está instalado)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

        # determinismo (más lento pero reproducible)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    except ImportError:
        pass  # no hay PyTorch, ignorar

    logger.info(f"Seeds fijadas en {seed}")


# Aplicación
SEED = 42
set_seeds(SEED)

2026-04-08 18:18:47,651 | INFO | Seeds fijadas en 42


# **10. Entrenamiento de modelo XGBoost**

## **10.1. Función unitaria por bundle**

In [ ]:
from xgboost import XGBClassifier
import numpy as np


def run_xgboost_for_bundle_seq2one(
    bundle,
    *,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight=None,
    verbose=False,
):
    """
    Ejecuta XGBoost para un bundle seq2one.

    - Usa TRAIN para fit
    - Predice en VALID y TEST
    - Devuelve predicciones
    - Soporta labels arbitrarias (ej. [-1, 0, 1]) mediante codificación interna

    Parámetros
    ----------
    class_weight : None | "balanced" | dict
        - None       -> entrenamiento natural
        - "balanced" -> pesos inversamente proporcionales a la frecuencia de clase
        - dict       -> pesos manuales por clase original, ej. {-1: 2.0, 0: 1.0, 1: 2.0}
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    X_test  = bundle["test"]["X"]

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)
    X_test_model  = prepare_X_for_model(X_test,  input_mode=input_mode)

    # =========================
    # 3. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 4. SAMPLE WEIGHTS
    # =========================
    sample_weight = None

    if class_weight is None:
        sample_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        weights_by_idx = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }
        sample_weight = np.array(
            [weights_by_idx[idx] for idx in y_train_enc],
            dtype=np.float32,
        )

    elif isinstance(class_weight, dict):
        weights_by_idx = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }
        sample_weight = np.array(
            [weights_by_idx.get(idx, 1.0) for idx in y_train_enc],
            dtype=np.float32,
        )

    else:
        raise ValueError(
            "class_weight debe ser None, 'balanced' o dict"
        )

    # =========================
    # 5. MODELO
    # =========================
    model = XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=random_state,
        n_jobs=n_jobs,
        eval_metric="mlogloss",
        verbosity=1 if verbose else 0,
    )

    # =========================
    # 6. TRAIN
    # =========================
    model.fit(X_train_model, y_train_enc, sample_weight=sample_weight)

    # =========================
    # 7. PREDICT
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_test_enc  = model.predict(X_test_model)

    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])
    y_pred_test  = np.array([idx_to_class[int(y)] for y in y_pred_test_enc])

    y_proba_valid = model.predict_proba(X_valid_model)
    y_proba_test  = model.predict_proba(X_test_model)

    return {
        "model": model,
        "classes_": classes_,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "class_weight": class_weight,
        "sample_weight": sample_weight,
        "y_pred_valid": y_pred_valid,
        "y_pred_test": y_pred_test,
        "y_proba_valid": y_proba_valid,
        "y_proba_test": y_proba_test,
    }

## **10.2. Función de evaluación sobre uno o más bundles**

In [ ]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_xgboost_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "xgboost",
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa XGBoost para uno o varios bundles seq2one y
    retorna un DataFrame consolidado.

    Incluye soporte para clases desbalanceadas o balanceadas.

    class_weight:
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática
        - dict       -> pesos manuales
    """

    # --------------------------------------------------
    # 1) Normalizar entrada a lista
    # --------------------------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    # --------------------------------------------------
    # 2) Validar split
    # --------------------------------------------------
    if split not in ("valid", "test"):
        raise ValueError("split debe ser 'valid' o 'test'")

    rows = []

    # --------------------------------------------------
    # 3) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 4) Entrenar + predecir
        # ----------------------------------------------
        preds = run_xgboost_for_bundle_seq2one(
            bundle,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=False,
        )

        # ----------------------------------------------
        # 5) Seleccionar y_true / y_pred del split
        # ----------------------------------------------
        y_true = bundle[split]["y"]
        y_pred_key = f"y_pred_{split}"

        if y_pred_key not in preds:
            raise KeyError(
                f"No existe '{y_pred_key}' en la salida de "
                f"run_xgboost_for_bundle_seq2one"
            )

        y_pred = preds[y_pred_key]

        # ----------------------------------------------
        # 6) Métricas de clasificación
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split=split,
            target=target,
            labels=[-1, 0, 1],
        )

        # ----------------------------------------------
        # 7) A DataFrame
        # ----------------------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split=split,
            window_size=window_size,
            target=target,
        )

        df_row["horizon_min"] = horizon
        df_row["class_weight_mode"] = class_weight

        rows.append(df_row)

    # --------------------------------------------------
    # 8) Consolidar salida
    # --------------------------------------------------
    return pd.concat(rows, ignore_index=True)

## **10.3. Función orquestadora por `window_size`**

In [ ]:
import gc
import pandas as pd


def run_xgboost(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "xgboost",
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta XGBoost para una sola window_size
    sobre los targets T2:
      - t2_dir_thr_90
      - t2_dir_thr_120

    Retorna un DataFrame consolidado con métricas de VALID y TEST.

    class_weight : None | "balanced" | dict
    """

    size = int(window_size)

    bundle_t2_90 = bundle_t2_120 = None
    bundles_t2 = None
    df_out = None

    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"XGBOOST | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"class_weight = {class_weight}")

        # --------------------------------------------------
        # 2) Construcción de bundles
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['t2_dir_thr_90', 't2_dir_thr_120']")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90, bundle_t2_120]

        # --------------------------------------------------
        # 3) Evaluación
        # --------------------------------------------------
        dfs = []

        for split in ["valid", "test"]:
            if verbose:
                print(
                    f"\n[EVAL] L{size} | split={split} | "
                    f"model={model_name_effective} | class_weight={class_weight}"
                )

            df_split = eval_xgboost_bundles(
                bundles_t2,
                split=split,
                model_name=model_name_effective,
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                random_state=random_state,
                n_jobs=n_jobs,
                input_mode=input_mode,
                class_weight=class_weight,
                verbose=verbose,
            )
            dfs.append(df_split)

        # --------------------------------------------------
        # 4) Consolidación
        # --------------------------------------------------
        df_out = (
            pd.concat(dfs, ignore_index=True)
            .sort_values(["window_size", "target", "split", "horizon_min", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "split",
                        "target",
                        "model",
                        "horizon_min",
                        "class_weight_mode",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .sort_values(
                    ["split", "target", "model", "horizon_min", "class_weight_mode"]
                )
                .to_string(index=False)
            )

        return df_out

    finally:
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()

## **10.4. Función incremental multi-ventana**

In [ ]:
from pathlib import Path
import pandas as pd


def run_xgboost_incremental(
    *,
    window_sizes: list[int],
    name: str = "xgboost",
    verbose: bool = True,
    n_estimators: int = 100,
    max_depth: int = 6,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 1.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta XGBoost de forma incremental para múltiples window_sizes.

    - Carga histórico si existe
    - Hace SKIP si un window_size ya está completo
    - Corre run_xgboost(window_size=L) para los faltantes
    - Agrega resultados nuevos al histórico
    - Guarda usando save_classification_metrics(df_hist, name=name)

    class_weight : None | "balanced" | dict
    """

    # nombre efectivo del experimento
    name_effective = f"{name}_balanced" if class_weight == "balanced" else name

    metrics_dir = DRIVE_DIR / "metrics/classification_metrics"
    metrics_path = metrics_dir / f"classification_{name_effective}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico si existe
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definir completitud esperada por window_size
    # --------------------------------------------------
    expected_targets = {"t2_dir_thr_90", "t2_dir_thr_120"}
    expected_splits = {"valid", "test"}
    expected_models = {name_effective}

    # --------------------------------------------------
    # 3) Iterar por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)

        # ----------------------------------------------
        # Skip robusto por window_size
        # ----------------------------------------------
        if not df_hist.empty:
            dfL = df_hist[df_hist["window_size"] == L]

            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits = set(dfL["split"].unique()) if not dfL.empty else set()
            done_models = set(dfL["model"].unique()) if not dfL.empty else set()

            if class_weight == "balanced":
                done_class_weight_mode = (
                    set(dfL["class_weight_mode"].dropna().unique())
                    if ("class_weight_mode" in dfL.columns and not dfL.empty)
                    else set()
                )

                is_complete = (
                    expected_targets.issubset(done_targets)
                    and expected_splits.issubset(done_splits)
                    and expected_models.issubset(done_models)
                    and {"balanced"}.issubset(done_class_weight_mode)
                )
            else:
                is_complete = (
                    expected_targets.issubset(done_targets)
                    and expected_splits.issubset(done_splits)
                    and expected_models.issubset(done_models)
                )

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name_effective} L={L} ya existe completo en Drive")
                continue

        # ----------------------------------------------
        # Ejecutar XGBoost para este window_size
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 80)
            print(f"[RUN] {name_effective} | L={L} | class_weight={class_weight}")
            print("-" * 80)

        df_L = run_xgboost(
            window_size=L,
            verbose=verbose,
            model_name=name,
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
        )

        # Etiqueta de familia
        df_L["family"] = name_effective

        # ----------------------------------------------
        # Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # Eliminar duplicados por seguridad
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
        ]
        if "class_weight_mode" in df_hist.columns:
            subset_cols.append("class_weight_mode")

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # Guardar histórico actualizado
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name_effective)

    # --------------------------------------------------
    # 4) Retorno final ordenado
    # --------------------------------------------------
    sort_cols = ["window_size", "target", "split", "horizon_min", "model"]
    if "class_weight_mode" in df_hist.columns:
        sort_cols.append("class_weight_mode")

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **10.5. Ejecución final del experimento**

In [ ]:
df_xgboost_none = run_xgboost_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="xgboost",
    verbose=True,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight=None,
)

[SKIP] xgboost L=30 ya existe completo en Drive
[SKIP] xgboost L=60 ya existe completo en Drive
[SKIP] xgboost L=90 ya existe completo en Drive
[SKIP] xgboost L=120 ya existe completo en Drive
[SKIP] xgboost L=180 ya existe completo en Drive


In [ ]:
df_xgboost_balanced = run_xgboost_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="xgboost",
    verbose=True,
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight="balanced",
)

[SKIP] xgboost_balanced L=30 ya existe completo en Drive
[SKIP] xgboost_balanced L=60 ya existe completo en Drive
[SKIP] xgboost_balanced L=90 ya existe completo en Drive
[SKIP] xgboost_balanced L=120 ya existe completo en Drive
[SKIP] xgboost_balanced L=180 ya existe completo en Drive


# **11. Entrenamiento de modelo LigthGBM**

## **11.1. Función unitaria por bundle**

In [ ]:
from lightgbm import LGBMClassifier
import numpy as np


def run_lightgbm_for_bundle_seq2one(
    bundle,
    *,
    n_estimators=100,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.1,
    subsample=1.0,
    colsample_bytree=1.0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=42,
    n_jobs=-1,
    input_mode="2d_flat",
    class_weight=None,
    verbose=False,
):
    """
    Ejecuta LightGBM para un bundle seq2one.

    - Usa TRAIN para fit
    - Predice en VALID y TEST
    - Devuelve predicciones
    - Soporta labels arbitrarias (ej. [-1, 0, 1]) mediante codificación interna

    Parámetros
    ----------
    class_weight : None | "balanced" | dict
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática por frecuencia inversa
        - dict       -> pesos manuales por clase original, ej. {-1: 2.0, 0: 1.0, 1: 2.0}
    """

    # =========================
    # 1. EXTRAER DATA
    # =========================
    X_train = bundle["train"]["X"]
    y_train = bundle["train"]["y"]

    X_valid = bundle["valid"]["X"]
    X_test  = bundle["test"]["X"]

    # =========================
    # 2. PREPARAR INPUT
    # =========================
    X_train_model = prepare_X_for_model(X_train, input_mode=input_mode)
    X_valid_model = prepare_X_for_model(X_valid, input_mode=input_mode)
    X_test_model  = prepare_X_for_model(X_test,  input_mode=input_mode)

    # =========================
    # 3. CODIFICAR LABELS
    # =========================
    classes_ = np.sort(np.unique(y_train))
    class_to_idx = {cls: idx for idx, cls in enumerate(classes_)}
    idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

    y_train_enc = np.array([class_to_idx[y] for y in y_train], dtype=np.int32)
    num_class = len(classes_)

    # =========================
    # 4. CLASS WEIGHT
    # =========================
    lgbm_class_weight = None

    if class_weight is None:
        lgbm_class_weight = None

    elif class_weight == "balanced":
        counts = np.bincount(y_train_enc, minlength=num_class)
        total = counts.sum()

        lgbm_class_weight = {
            idx: total / (num_class * count)
            for idx, count in enumerate(counts)
        }

    elif isinstance(class_weight, dict):
        lgbm_class_weight = {
            class_to_idx[cls]: weight
            for cls, weight in class_weight.items()
            if cls in class_to_idx
        }

    else:
        raise ValueError(
            "class_weight debe ser None, 'balanced' o dict"
        )

    # =========================
    # 5. MODELO
    # =========================
    model = LGBMClassifier(
        objective="multiclass",
        num_class=num_class,
        n_estimators=n_estimators,
        max_depth=max_depth,
        num_leaves=num_leaves,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=random_state,
        n_jobs=n_jobs,
        class_weight=lgbm_class_weight,
        verbosity=1 if verbose else -1,
    )

    # =========================
    # 6. TRAIN
    # =========================
    model.fit(X_train_model, y_train_enc)

    # =========================
    # 7. PREDICT
    # =========================
    y_pred_valid_enc = model.predict(X_valid_model)
    y_pred_test_enc  = model.predict(X_test_model)

    y_pred_valid = np.array([idx_to_class[int(y)] for y in y_pred_valid_enc])
    y_pred_test  = np.array([idx_to_class[int(y)] for y in y_pred_test_enc])

    y_proba_valid = model.predict_proba(X_valid_model)
    y_proba_test  = model.predict_proba(X_test_model)

    return {
        "model": model,
        "classes_": classes_,
        "class_to_idx": class_to_idx,
        "idx_to_class": idx_to_class,
        "class_weight": class_weight,
        "lgbm_class_weight": lgbm_class_weight,
        "y_pred_valid": y_pred_valid,
        "y_pred_test": y_pred_test,
        "y_proba_valid": y_proba_valid,
        "y_proba_test": y_proba_test,
    }

## **11.2. Función de evaluación sobre uno o más bundles**

In [ ]:
from typing import Any, Dict, List, Sequence, Union
import pandas as pd


def eval_lightgbm_bundles(
    bundles: Union[Dict[str, Any], Sequence[Dict[str, Any]]],
    *,
    split: str = "valid",
    model_name: str = "lightgbm",
    n_estimators: int = 100,
    max_depth: int = -1,
    num_leaves: int = 31,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 0.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
    verbose: bool = False,
) -> pd.DataFrame:
    """
    Evalúa LightGBM para uno o varios bundles seq2one y
    retorna un DataFrame consolidado.

    Incluye soporte para clases desbalanceadas o balanceadas.

    class_weight:
        - None       -> entrenamiento natural
        - "balanced" -> ponderación automática
        - dict       -> pesos manuales
    """

    # --------------------------------------------------
    # 1) Normalizar entrada a lista
    # --------------------------------------------------
    if isinstance(bundles, dict):
        bundles_list: List[Dict[str, Any]] = [bundles]
    else:
        bundles_list = list(bundles)

    # --------------------------------------------------
    # 2) Validar split
    # --------------------------------------------------
    if split not in ("valid", "test"):
        raise ValueError("split debe ser 'valid' o 'test'")

    rows = []

    # --------------------------------------------------
    # 3) Iterar por bundles
    # --------------------------------------------------
    for bundle in bundles_list:
        target = bundle.get("target", None)
        window_size = int(bundle["window_size"])
        horizon = int(bundle.get("horizon", -1))

        if verbose:
            print(
                f"  -> L{window_size} | "
                f"target={target} | "
                f"split={split} | "
                f"model={model_name} | "
                f"class_weight={class_weight}"
            )

        # ----------------------------------------------
        # 4) Entrenar + predecir
        # ----------------------------------------------
        preds = run_lightgbm_for_bundle_seq2one(
            bundle,
            n_estimators=n_estimators,
            max_depth=max_depth,
            num_leaves=num_leaves,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
            verbose=False,
        )

        # ----------------------------------------------
        # 5) Seleccionar y_true / y_pred del split
        # ----------------------------------------------
        y_true = bundle[split]["y"]
        y_pred_key = f"y_pred_{split}"

        if y_pred_key not in preds:
            raise KeyError(
                f"No existe '{y_pred_key}' en la salida de "
                f"run_lightgbm_for_bundle_seq2one"
            )

        y_pred = preds[y_pred_key]

        # ----------------------------------------------
        # 6) Métricas de clasificación
        # ----------------------------------------------
        metrics = compute_classification_metrics(
            y_true=y_true,
            y_pred=y_pred,
            model_name=model_name,
            split=split,
            target=target,
            labels=[-1, 0, 1],
        )

        # ----------------------------------------------
        # 7) A DataFrame
        # ----------------------------------------------
        df_row = metrics_to_df(
            metrics,
            model=model_name,
            split=split,
            window_size=window_size,
            target=target,
        )

        df_row["horizon_min"] = horizon
        df_row["class_weight_mode"] = class_weight

        rows.append(df_row)

    # --------------------------------------------------
    # 8) Consolidar salida
    # --------------------------------------------------
    return pd.concat(rows, ignore_index=True)

## **11.3. Función orquestadora por `window_size`**

In [ ]:
import gc
import pandas as pd


def run_lightgbm(
    window_size: int,
    *,
    verbose: bool = True,
    model_name: str = "lightgbm",
    n_estimators: int = 100,
    max_depth: int = -1,
    num_leaves: int = 31,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 0.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta LightGBM para una sola window_size
    sobre los targets T2:
      - t2_dir_thr_90
      - t2_dir_thr_120

    Retorna un DataFrame consolidado con métricas de VALID y TEST.

    class_weight : None | "balanced" | dict
    """

    size = int(window_size)

    bundle_t2_90 = bundle_t2_120 = None
    bundles_t2 = None
    df_out = None

    model_name_effective = (
        f"{model_name}_balanced" if class_weight == "balanced" else model_name
    )

    try:
        # --------------------------------------------------
        # 1) Encabezado
        # --------------------------------------------------
        if verbose:
            print("\n" + "=" * 80)
            print(f"LIGHTGBM | T2 SEQ2ONE | WINDOW_SIZE=L{size}")
            print("=" * 80)
            print(f"class_weight = {class_weight}")

        # --------------------------------------------------
        # 2) Construcción de bundles para esta ventana
        # --------------------------------------------------
        if verbose:
            print(f"\n[BUILD] L{size} | targets = ['t2_dir_thr_90', 't2_dir_thr_120']")

        bundle_t2_90, bundle_t2_120 = create_bundles(
            window_size=size,
            targets=["t2_dir_thr_90", "t2_dir_thr_120"],
            windows_paths=WINDOWS_PATHS,
            scaler_path=SCALER_T2_PATH,
        )

        bundles_t2 = [bundle_t2_90, bundle_t2_120]

        # --------------------------------------------------
        # 3) Evaluación por split
        # --------------------------------------------------
        dfs = []

        for split in ["valid", "test"]:
            if verbose:
                print(
                    f"\n[EVAL] L{size} | split={split} | "
                    f"model={model_name_effective} | class_weight={class_weight}"
                )

            df_split = eval_lightgbm_bundles(
                bundles_t2,
                split=split,
                model_name=model_name_effective,
                n_estimators=n_estimators,
                max_depth=max_depth,
                num_leaves=num_leaves,
                learning_rate=learning_rate,
                subsample=subsample,
                colsample_bytree=colsample_bytree,
                reg_alpha=reg_alpha,
                reg_lambda=reg_lambda,
                random_state=random_state,
                n_jobs=n_jobs,
                input_mode=input_mode,
                class_weight=class_weight,
                verbose=verbose,
            )
            dfs.append(df_split)

        # --------------------------------------------------
        # 4) Consolidación final
        # --------------------------------------------------
        df_out = (
            pd.concat(dfs, ignore_index=True)
            .sort_values(["window_size", "target", "split", "horizon_min", "model"])
            .reset_index(drop=True)
        )

        # --------------------------------------------------
        # 5) Resumen final
        # --------------------------------------------------
        if verbose:
            print(f"\n[DONE] L{size} | rows={len(df_out)}")
            print(
                df_out[
                    [
                        "window_size",
                        "split",
                        "target",
                        "model",
                        "horizon_min",
                        "class_weight_mode",
                        "balanced_accuracy",
                        "f1_macro",
                    ]
                ]
                .sort_values(
                    ["split", "target", "model", "horizon_min", "class_weight_mode"]
                )
                .to_string(index=False)
            )

        return df_out

    finally:
        # --------------------------------------------------
        # 6) Liberación de memoria
        # --------------------------------------------------
        del bundle_t2_90, bundle_t2_120, bundles_t2
        gc.collect()

## **11.4. Función incremental multi-ventana**

In [ ]:
from pathlib import Path
import pandas as pd


def run_lightgbm_incremental(
    *,
    window_sizes: list[int],
    name: str = "lightgbm",
    verbose: bool = True,
    n_estimators: int = 100,
    max_depth: int = -1,
    num_leaves: int = 31,
    learning_rate: float = 0.1,
    subsample: float = 1.0,
    colsample_bytree: float = 1.0,
    reg_alpha: float = 0.0,
    reg_lambda: float = 0.0,
    random_state: int = 42,
    n_jobs: int = -1,
    input_mode: str = "2d_flat",
    class_weight=None,
) -> pd.DataFrame:
    """
    Ejecuta LightGBM de forma incremental para múltiples window_sizes.

    - Carga histórico si existe
    - Hace SKIP si un window_size ya está completo
    - Corre run_lightgbm(window_size=L) para los faltantes
    - Agrega resultados nuevos al histórico
    - Guarda usando save_classification_metrics(df_hist, name=name)

    class_weight : None | "balanced" | dict
    """

    # nombre efectivo del experimento
    name_effective = f"{name}_balanced" if class_weight == "balanced" else name

    metrics_dir = DRIVE_DIR / "metrics/classification_metrics"
    metrics_path = metrics_dir / f"classification_{name_effective}_metrics.parquet"

    # --------------------------------------------------
    # 1) Cargar histórico si existe
    # --------------------------------------------------
    if metrics_path.exists():
        df_hist = pd.read_parquet(metrics_path)
    else:
        df_hist = pd.DataFrame()

    # --------------------------------------------------
    # 2) Definir completitud esperada por window_size
    # --------------------------------------------------
    expected_targets = {"t2_dir_thr_90", "t2_dir_thr_120"}
    expected_splits = {"valid", "test"}
    expected_models = {name_effective}

    # --------------------------------------------------
    # 3) Iterar por window_sizes
    # --------------------------------------------------
    for L in window_sizes:
        L = int(L)

        # ----------------------------------------------
        # Skip robusto por window_size
        # ----------------------------------------------
        if not df_hist.empty:
            dfL = df_hist[df_hist["window_size"] == L]

            done_targets = set(dfL["target"].unique()) if not dfL.empty else set()
            done_splits = set(dfL["split"].unique()) if not dfL.empty else set()
            done_models = set(dfL["model"].unique()) if not dfL.empty else set()

            if class_weight == "balanced":
                done_class_weight_mode = (
                    set(dfL["class_weight_mode"].dropna().unique())
                    if ("class_weight_mode" in dfL.columns and not dfL.empty)
                    else set()
                )

                is_complete = (
                    expected_targets.issubset(done_targets)
                    and expected_splits.issubset(done_splits)
                    and expected_models.issubset(done_models)
                    and {"balanced"}.issubset(done_class_weight_mode)
                )
            else:
                is_complete = (
                    expected_targets.issubset(done_targets)
                    and expected_splits.issubset(done_splits)
                    and expected_models.issubset(done_models)
                )

            if is_complete:
                if verbose:
                    print(f"[SKIP] {name_effective} L={L} ya existe completo en Drive")
                continue

        # ----------------------------------------------
        # Ejecutar LightGBM para este window_size
        # ----------------------------------------------
        if verbose:
            print("\n" + "-" * 80)
            print(f"[RUN] {name_effective} | L={L} | class_weight={class_weight}")
            print("-" * 80)

        df_L = run_lightgbm(
            window_size=L,
            verbose=verbose,
            model_name=name,
            n_estimators=n_estimators,
            max_depth=max_depth,
            num_leaves=num_leaves,
            learning_rate=learning_rate,
            subsample=subsample,
            colsample_bytree=colsample_bytree,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            random_state=random_state,
            n_jobs=n_jobs,
            input_mode=input_mode,
            class_weight=class_weight,
        )

        # Etiqueta de familia
        df_L["family"] = name_effective

        # ----------------------------------------------
        # Actualizar histórico
        # ----------------------------------------------
        if df_hist.empty:
            df_hist = df_L.copy()
        else:
            df_hist = pd.concat([df_hist, df_L], ignore_index=True)

        # ----------------------------------------------
        # Eliminar duplicados por seguridad
        # ----------------------------------------------
        subset_cols = [
            "window_size",
            "target",
            "split",
            "model",
            "horizon_min",
        ]
        if "class_weight_mode" in df_hist.columns:
            subset_cols.append("class_weight_mode")

        df_hist = (
            df_hist
            .drop_duplicates(subset=subset_cols, keep="last")
            .reset_index(drop=True)
        )

        # ----------------------------------------------
        # Guardar histórico actualizado
        # ----------------------------------------------
        save_classification_metrics(df_hist, name=name_effective)

    # --------------------------------------------------
    # 4) Retorno final ordenado
    # --------------------------------------------------
    sort_cols = ["window_size", "target", "split", "horizon_min", "model"]
    if "class_weight_mode" in df_hist.columns:
        sort_cols.append("class_weight_mode")

    return df_hist.sort_values(sort_cols).reset_index(drop=True)

## **11.5. Ejecución final del experimento**

In [ ]:
df_lightgbm_none = run_lightgbm_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="lightgbm",
    class_weight=None,
)

df_lightgbm_balanced = run_lightgbm_incremental(
    window_sizes=[30, 60, 90, 120, 180],
    name="lightgbm",
    class_weight="balanced",
)

[SKIP] lightgbm L=30 ya existe completo en Drive
[SKIP] lightgbm L=60 ya existe completo en Drive
[SKIP] lightgbm L=90 ya existe completo en Drive
[SKIP] lightgbm L=120 ya existe completo en Drive
[SKIP] lightgbm L=180 ya existe completo en Drive
[SKIP] lightgbm_balanced L=30 ya existe completo en Drive
[SKIP] lightgbm_balanced L=60 ya existe completo en Drive
[SKIP] lightgbm_balanced L=90 ya existe completo en Drive
[SKIP] lightgbm_balanced L=120 ya existe completo en Drive
[SKIP] lightgbm_balanced L=180 ya existe completo en Drive


In [ ]:
df_xgboost_none

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,xgboost,test,30,t2_dir_thr_120,93990,0.345481,0.270391,0.438021,0.580530,0.529032,0.345481,0.333333,0.012148,120,None,xgboost
1,xgboost,valid,30,t2_dir_thr_120,93508,0.335749,0.285607,0.605468,0.719297,0.411436,0.335749,0.333333,0.002415,120,None,xgboost
2,xgboost,test,30,t2_dir_thr_90,93990,0.346949,0.274628,0.440230,0.579849,0.523586,0.346949,0.333333,0.013616,90,None,xgboost
3,xgboost,valid,30,t2_dir_thr_90,93508,0.338178,0.289271,0.599838,0.713982,0.475752,0.338178,0.333333,0.004844,90,None,xgboost
4,xgboost,test,60,t2_dir_thr_120,88140,0.345466,0.267808,0.421657,0.564727,0.455447,0.345466,0.333333,0.012132,120,None,xgboost
5,xgboost,valid,60,t2_dir_thr_120,87688,0.335489,0.281885,0.585355,0.703871,0.411697,0.335489,0.333333,0.002156,120,None,xgboost
6,xgboost,test,60,t2_dir_thr_90,88140,0.348957,0.275718,0.426980,0.565963,0.499741,0.348957,0.333333,0.015624,90,None,xgboost
7,xgboost,valid,60,t2_dir_thr_90,87688,0.337462,0.284190,0.578828,0.698271,0.458207,0.337462,0.333333,0.004129,90,None,xgboost
8,xgboost,test,90,t2_dir_thr_120,82290,0.344968,0.263192,0.403664,0.548718,0.429409,0.344968,0.333333,0.011635,120,None,xgboost
9,xgboost,valid,90,t2_dir_thr_120,81868,0.335331,0.279336,0.575392,0.696731,0.392976,0.335331,0.333333,0.001998,120,None,xgboost


In [ ]:
df_xgboost_balanced

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,xgboost_balanced,test,30,t2_dir_thr_120,93990,0.433425,0.434117,0.533553,0.540430,0.435844,0.433425,0.333333,0.100092,120,balanced,xgboost_balanced
1,xgboost_balanced,valid,30,t2_dir_thr_120,93508,0.411931,0.416423,0.642929,0.660692,0.426417,0.411931,0.333333,0.078597,120,balanced,xgboost_balanced
2,xgboost_balanced,test,30,t2_dir_thr_90,93990,0.438932,0.438846,0.535633,0.537695,0.439061,0.438932,0.333333,0.105599,90,balanced,xgboost_balanced
3,xgboost_balanced,valid,30,t2_dir_thr_90,93508,0.432357,0.435098,0.643454,0.651057,0.439086,0.432357,0.333333,0.099024,90,balanced,xgboost_balanced
4,xgboost_balanced,test,60,t2_dir_thr_120,88140,0.427193,0.427777,0.516507,0.520161,0.428910,0.427193,0.333333,0.093859,120,balanced,xgboost_balanced
5,xgboost_balanced,valid,60,t2_dir_thr_120,87688,0.406241,0.410017,0.624033,0.643737,0.420428,0.406241,0.333333,0.072908,120,balanced,xgboost_balanced
6,xgboost_balanced,test,60,t2_dir_thr_90,88140,0.434348,0.434758,0.522911,0.526651,0.435498,0.434348,0.333333,0.101014,90,balanced,xgboost_balanced
7,xgboost_balanced,valid,60,t2_dir_thr_90,87688,0.421532,0.425245,0.625993,0.639848,0.432433,0.421532,0.333333,0.088199,90,balanced,xgboost_balanced
8,xgboost_balanced,test,90,t2_dir_thr_120,82290,0.418677,0.419037,0.498818,0.502503,0.420470,0.418677,0.333333,0.085344,120,balanced,xgboost_balanced
9,xgboost_balanced,valid,90,t2_dir_thr_120,81868,0.406924,0.411116,0.618057,0.640360,0.424354,0.406924,0.333333,0.073591,120,balanced,xgboost_balanced


In [ ]:
df_lightgbm_none

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,lightgbm,test,30,t2_dir_thr_120,93990,0.346007,0.270870,0.438484,0.581200,0.539025,0.346007,0.333333,0.012674,120,None,lightgbm
1,lightgbm,valid,30,t2_dir_thr_120,93508,0.335822,0.285684,0.605465,0.719382,0.434601,0.335822,0.333333,0.002488,120,None,lightgbm
2,lightgbm,test,30,t2_dir_thr_90,93990,0.346059,0.272717,0.439069,0.579402,0.483377,0.346059,0.333333,0.012725,90,None,lightgbm
3,lightgbm,valid,30,t2_dir_thr_90,93508,0.338557,0.289991,0.600203,0.714131,0.447900,0.338557,0.333333,0.005223,90,None,lightgbm
4,lightgbm,test,60,t2_dir_thr_120,88140,0.346522,0.269295,0.422812,0.565759,0.468972,0.346522,0.333333,0.013189,120,None,lightgbm
5,lightgbm,valid,60,t2_dir_thr_120,87688,0.335365,0.281405,0.585276,0.704087,0.402131,0.335365,0.333333,0.002032,120,None,lightgbm
6,lightgbm,test,60,t2_dir_thr_90,88140,0.348673,0.275155,0.426468,0.565680,0.492274,0.348673,0.333333,0.015340,90,None,lightgbm
7,lightgbm,valid,60,t2_dir_thr_90,87688,0.337283,0.284400,0.578697,0.697587,0.435110,0.337283,0.333333,0.003950,90,None,lightgbm
8,lightgbm,test,90,t2_dir_thr_120,82290,0.344566,0.262034,0.402672,0.548584,0.437680,0.344566,0.333333,0.011232,120,None,lightgbm
9,lightgbm,valid,90,t2_dir_thr_120,81868,0.335174,0.279101,0.575138,0.696524,0.388523,0.335174,0.333333,0.001841,120,None,lightgbm


In [ ]:
df_lightgbm_balanced

,model,split,window_size,target,n_samples,balanced_accuracy,f1_macro,f1_weighted,accuracy,precision_macro,recall_macro,balanced_accuracy_naive,bal_acc_gain_vs_naive,horizon_min,class_weight_mode,family
0,lightgbm_balanced,test,30,t2_dir_thr_120,93990,0.437305,0.437990,0.536303,0.542909,0.439697,0.437305,0.333333,0.103971,120,balanced,lightgbm_balanced
1,lightgbm_balanced,valid,30,t2_dir_thr_120,93508,0.412575,0.416917,0.643373,0.660585,0.426311,0.412575,0.333333,0.079242,120,balanced,lightgbm_balanced
2,lightgbm_balanced,test,30,t2_dir_thr_90,93990,0.437927,0.437943,0.534988,0.536823,0.438141,0.437927,0.333333,0.104594,90,balanced,lightgbm_balanced
3,lightgbm_balanced,valid,30,t2_dir_thr_90,93508,0.429279,0.431511,0.641406,0.649142,0.435507,0.429279,0.333333,0.095946,90,balanced,lightgbm_balanced
4,lightgbm_balanced,test,60,t2_dir_thr_120,88140,0.429492,0.430073,0.518212,0.522079,0.431099,0.429492,0.333333,0.096158,120,balanced,lightgbm_balanced
5,lightgbm_balanced,valid,60,t2_dir_thr_120,87688,0.404536,0.408075,0.622300,0.641696,0.418020,0.404536,0.333333,0.071202,120,balanced,lightgbm_balanced
6,lightgbm_balanced,test,60,t2_dir_thr_90,88140,0.434461,0.434706,0.522367,0.526027,0.435343,0.434461,0.333333,0.101128,90,balanced,lightgbm_balanced
7,lightgbm_balanced,valid,60,t2_dir_thr_90,87688,0.422203,0.425961,0.626227,0.639620,0.432890,0.422203,0.333333,0.088870,90,balanced,lightgbm_balanced
8,lightgbm_balanced,test,90,t2_dir_thr_120,82290,0.418983,0.419180,0.499344,0.502844,0.420626,0.418983,0.333333,0.085650,120,balanced,lightgbm_balanced
9,lightgbm_balanced,valid,90,t2_dir_thr_120,81868,0.405221,0.408877,0.616592,0.638051,0.421140,0.405221,0.333333,0.071888,120,balanced,lightgbm_balanced
